# Institutional Stochastic Forecasting Engine — Nasdaq (QQQ)

**A probabilistic forecasting system built entirely on stochastic processes, state-space modelling, probability theory and regime modelling.**

> No technical indicators. No RSI. No MACD. No moving-average crossovers.
> Every signal is derived from probability, latent-state inference and Monte Carlo.

---

### Architecture

| Module | Component | Method |
|---|---|---|
| 0 | Data & Stochastic Features | log-returns, realized vol, entropy, Hurst, fractal dim, vol-of-vol |
| 1 | Markov Chain Engine | 1st / 2nd / 3rd order + Variable-Length Markov Chain |
| 2 | Hidden Markov Model | Gaussian HMM, BIC/AIC selection (2–6 states) |
| 3 | Hidden Semi-Markov Model | explicit-duration HSMM |
| 4 | Markov-Switching Volatility | 4-regime switching variance |
| 5 | Kalman Filter | KF / EKF / UKF state-space |
| 6 | Particle Filter | Sequential Monte Carlo (1k / 5k / 10k particles) |
| 7 | Bayesian Update Engine | daily posterior regime filtering |
| 8 | Monte Carlo Forecasting | 1k / 5k / 10k paths, 1–60 day horizons |
| 9 | Ensemble Model | probability-weighted combination |
| 10 | Backtesting | probability-only signals, full metric suite |
| 11 | Robustness Testing | walk-forward, bootstrap, calibration, stress |
| 12 | Research Report | auto-generated report + forecast dashboard |

*Designed as a quant hedge-fund research artefact. Runs end-to-end in Google Colab.*


## Module 0 — Environment, Data & Stochastic Feature Engineering

In [ ]:
# === Dependencies (Colab-ready) =============================================
# Quietly install the scientific stack. Safe to re-run.
import sys, subprocess

def _pip(pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)

_pip(["yfinance", "hmmlearn", "statsmodels", "arch", "scipy", "numpy",
      "pandas", "matplotlib", "seaborn", "scikit-learn"])
print("Dependencies ready.")


In [ ]:
# === Imports & global config ================================================
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats, linalg
from scipy.special import logsumexp
from numpy.random import default_rng

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (13, 5)
plt.rcParams["axes.grid"] = True
RNG = default_rng(7)
np.set_printoptions(precision=4, suppress=True)

TICKER   = "QQQ"
START    = "2000-01-01"
RF_ANNUAL = 0.02          # risk-free for ratios
ANN      = 252            # trading days / year
print("Config loaded. Forecasting engine target:", TICKER)


In [ ]:
# === Data acquisition (with deterministic fallback) =========================
# Primary: download daily QQQ from 2000 to today via yfinance.
# Fallback: if the network is unavailable, synthesise a regime-switching
# series so the entire notebook still runs end-to-end and is reproducible.
import datetime as dt

def load_prices(ticker=TICKER, start=START):
    try:
        import yfinance as yf
        df = yf.download(ticker, start=start, progress=False, auto_adjust=True)
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)
        px = df["Close"].dropna()
        if len(px) > 500:
            print(f"Downloaded {len(px)} live bars for {ticker} "
                  f"[{px.index[0].date()} -> {px.index[-1].date()}]")
            return px.astype(float), True
    except Exception as e:
        print("yfinance unavailable ->", e)
    # ---- Synthetic regime-switching fallback (GBM with Markov vol regimes) --
    print("Falling back to synthetic regime-switching QQQ proxy.")
    n = 6300
    idx = pd.bdate_range("2000-01-03", periods=n)
    rng = default_rng(42)
    mus  = np.array([0.0007, 0.0002, -0.0010, 0.0004])     # bull, calm, crash, recovery
    sigs = np.array([0.008,  0.012,  0.035,   0.018])
    P = np.array([[0.985,0.010,0.003,0.002],
                  [0.012,0.975,0.008,0.005],
                  [0.010,0.020,0.940,0.030],
                  [0.008,0.012,0.010,0.970]])
    s, rets = 0, []
    for _ in range(n):
        rets.append(rng.normal(mus[s], sigs[s]))
        s = rng.choice(4, p=P[s])
    px = pd.Series(100*np.exp(np.cumsum(rets)), index=idx, name="Close")
    return px, False

prices, IS_LIVE = load_prices()
prices.tail()


In [ ]:
# === Stochastic feature engineering =========================================
# All features are probabilistic / stochastic-process descriptors.
def hurst_rs(series, max_lag=64):
    # Rescaled-range (R/S) Hurst exponent.
    x = np.asarray(series, float)
    x = x[~np.isnan(x)]
    if len(x) < 32:
        return np.nan
    lags = np.unique(np.floor(np.logspace(0.7, np.log10(min(max_lag, len(x)//2)), 12)).astype(int))
    lags = lags[lags >= 4]
    rs = []
    for lag in lags:
        n_chunks = len(x)//lag
        if n_chunks < 1:
            continue
        vals = []
        for i in range(n_chunks):
            seg = x[i*lag:(i+1)*lag]
            z = seg - seg.mean()
            Z = np.cumsum(z)
            R = Z.max() - Z.min()
            S = seg.std(ddof=1)
            if S > 0:
                vals.append(R/S)
        if vals:
            rs.append((lag, np.mean(vals)))
    if len(rs) < 4:
        return np.nan
    lg = np.log(np.array([r[0] for r in rs]))
    lr = np.log(np.array([r[1] for r in rs]))
    H = np.polyfit(lg, lr, 1)[0]
    return float(np.clip(H, 0, 1))

def higuchi_fd(series, kmax=10):
    # Higuchi fractal dimension of a 1-D signal.
    x = np.asarray(series, float); x = x[~np.isnan(x)]
    N = len(x)
    if N < kmax*3:
        return np.nan
    Lk = []
    for k in range(1, kmax+1):
        Lm = []
        for m in range(k):
            idx = np.arange(m, N, k)
            if len(idx) < 2:
                continue
            ll = np.sum(np.abs(np.diff(x[idx])))*(N-1)/(((len(idx)-1))*k)
            Lm.append(ll)
        if Lm:
            Lk.append((k, np.mean(Lm)))
    if len(Lk) < 4:
        return np.nan
    lk = np.log(1.0/np.array([a[0] for a in Lk]))
    lL = np.log(np.array([a[1] for a in Lk]))
    return float(np.polyfit(lk, lL, 1)[0])

def rolling_entropy(returns, window=63, bins=8):
    # Rolling Shannon entropy (nats) of the return distribution.
    out = np.full(len(returns), np.nan)
    r = returns.values
    for i in range(window, len(r)):
        seg = r[i-window:i]
        h, _ = np.histogram(seg, bins=bins)
        p = h/ h.sum()
        p = p[p>0]
        out[i] = -np.sum(p*np.log(p))
    return pd.Series(out, index=returns.index)

# Core series
df = pd.DataFrame({"price": prices})
df["logret"] = np.log(df["price"]).diff()
df = df.dropna()

# Realized volatility (rolling std of log-returns) and clustering proxy
df["rvol_21"]  = df["logret"].rolling(21).std()*np.sqrt(ANN)
df["rvol_63"]  = df["logret"].rolling(63).std()*np.sqrt(ANN)
df["vol_clustering"] = df["logret"].abs().rolling(21).corr(df["logret"].abs().shift(1))
df["vol_of_vol"] = df["rvol_21"].rolling(63).std()
df["entropy_63"] = rolling_entropy(df["logret"], 63, 8)

# Rolling Hurst & fractal dimension (expensive -> stride + ffill)
H, FD = np.full(len(df), np.nan), np.full(len(df), np.nan)
win = 252; stride = 5
for i in range(win, len(df), stride):
    seg = df["logret"].iloc[i-win:i]
    H[i]  = hurst_rs(seg)
    FD[i] = higuchi_fd(seg)
df["hurst"]   = pd.Series(H,  index=df.index).ffill()
df["fractal"] = pd.Series(FD, index=df.index).ffill()

df = df.dropna()
print("Feature frame:", df.shape)
df[["logret","rvol_21","vol_of_vol","entropy_63","hurst","fractal","vol_clustering"]].describe().T


In [ ]:
# === Visual diagnostic of stochastic features ===============================
fig, ax = plt.subplots(3, 2, figsize=(15, 11))
ax[0,0].plot(df.index, df["price"], color="navy"); ax[0,0].set_yscale("log")
ax[0,0].set_title("QQQ price (log scale)")
ax[0,1].plot(df.index, df["rvol_21"], color="crimson", lw=.8)
ax[0,1].set_title("Realized volatility (21d, annualized)")
ax[1,0].plot(df.index, df["vol_of_vol"], color="darkorange", lw=.8)
ax[1,0].set_title("Volatility of volatility")
ax[1,1].plot(df.index, df["entropy_63"], color="teal", lw=.8)
ax[1,1].set_title("Rolling return entropy (63d)")
ax[2,0].plot(df.index, df["hurst"], color="purple", lw=.9); ax[2,0].axhline(.5, ls="--", c="k")
ax[2,0].set_title("Hurst exponent (>.5 persistent, <.5 mean-reverting)")
ax[2,1].plot(df.index, df["fractal"], color="green", lw=.9)
ax[2,1].set_title("Higuchi fractal dimension")
plt.tight_layout(); plt.show()

print(f"Latest Hurst={df['hurst'].iloc[-1]:.3f}  Fractal={df['fractal'].iloc[-1]:.3f}  "
      f"RVol={df['rvol_21'].iloc[-1]:.3f}  VoV={df['vol_of_vol'].iloc[-1]:.4f}")


## Module 1 — Markov Chain Engine
First / second / third order Markov chains plus a **Variable-Length Markov Chain (VLMC)**.
Returns are discretised into 3 states (Down / Flat / Up) using volatility-scaled thresholds.

In [ ]:
# === State discretisation ===================================================
# 3 symbolic states from standardized returns -> {0:Down, 1:Flat, 2:Up}.
STATE_NAMES = {0: "Down", 1: "Flat", 2: "Up"}
z = (df["logret"] / df["logret"].rolling(21).std()).dropna()
thr = 0.4   # flat band in std units
sym = np.where(z < -thr, 0, np.where(z > thr, 2, 1))
states = pd.Series(sym, index=z.index, name="state")
df_m = df.loc[states.index].copy()
df_m["state"] = states
print("State distribution:", {STATE_NAMES[k]: int((states==k).sum()) for k in range(3)})


In [ ]:
# === N-th order transition matrices =========================================
from itertools import product
NS = 3

def build_transition(seq, order):
    # Returns dict: context-tuple -> prob vector over next state (Laplace smoothed)
    counts = {}
    for i in range(order, len(seq)):
        ctx = tuple(seq[i-order:i])
        nxt = seq[i]
        counts.setdefault(ctx, np.ones(NS))   # Laplace prior
        counts[ctx][nxt] += 1
    return {ctx: c/c.sum() for ctx, c in counts.items()}

seq = states.values
T1 = build_transition(seq, 1)
T2 = build_transition(seq, 2)
T3 = build_transition(seq, 3)

# First-order matrix as array for display
M1 = np.vstack([T1[(s,)] for s in range(NS)])
print("First-order transition matrix P(s_{t+1}|s_t):")
print(pd.DataFrame(M1, index=[STATE_NAMES[s] for s in range(NS)],
                   columns=[STATE_NAMES[s] for s in range(NS)]).round(3))
MARKOV = {"T1": T1, "T2": T2, "T3": T3}


In [ ]:
# === Variable-Length Markov Chain (context-tree, KL pruning) ================
# Build contexts up to max order; keep a longer context only if its predictive
# distribution diverges from its parent (shorter) context by > delta (KL).
class VLMC:
    def __init__(self, max_order=4, min_count=20, kl_delta=0.03):
        self.max_order, self.min_count, self.kl_delta = max_order, min_count, kl_delta
        self.tree = {}
    @staticmethod
    def _kl(p, q):
        return np.sum(p*np.log((p+1e-12)/(q+1e-12)))
    def fit(self, seq):
        raw = {}  # ctx -> count vector
        for L in range(0, self.max_order+1):
            for i in range(L, len(seq)):
                ctx = tuple(seq[i-L:i])
                raw.setdefault(ctx, np.ones(NS))
                raw[ctx][seq[i]] += 1
        # keep contexts that are informative vs their parent
        self.tree = {(): raw[()]/raw[()].sum()}
        for ctx, c in raw.items():
            if ctx == ():
                continue
            if c.sum() - NS < self.min_count:   # too few observations
                continue
            parent = ctx[1:]
            p = c/c.sum()
            q = raw.get(parent, raw[()]); q = q/q.sum()
            if self._kl(p, q) > self.kl_delta:
                self.tree[ctx] = p
        return self
    def predict(self, history):
        # longest matching context wins
        for L in range(min(self.max_order, len(history)), -1, -1):
            ctx = tuple(history[-L:]) if L > 0 else ()
            if ctx in self.tree:
                return self.tree[ctx]
        return self.tree[()]

vlmc = VLMC(max_order=5, min_count=25, kl_delta=0.02).fit(seq)
print(f"VLMC retained {len(vlmc.tree)} variable-length contexts "
      f"(vs {NS**1+NS**2+NS**3}=full 1-3 order contexts).")


In [ ]:
# === Predictive power comparison (out-of-sample log-loss & accuracy) ========
split = int(len(seq)*0.7)
train, test = seq[:split], seq[split:]

models = {
    "1st-order": build_transition(train, 1),
    "2nd-order": build_transition(train, 2),
    "3rd-order": build_transition(train, 3),
}
vlmc_tr = VLMC(5, 25, 0.02).fit(train)

def eval_fixed(T, order, full):
    ll, hit, n = 0.0, 0, 0
    for i in range(max(order, split), len(full)):
        ctx = tuple(full[i-order:i])
        p = T.get(ctx, np.ones(NS)/NS)
        ll += -np.log(p[full[i]]+1e-12); hit += (p.argmax()==full[i]); n += 1
    return ll/n, hit/n

def eval_vlmc(model, full):
    ll, hit, n = 0.0, 0, 0
    for i in range(split, len(full)):
        p = model.predict(full[:i])
        ll += -np.log(p[full[i]]+1e-12); hit += (p.argmax()==full[i]); n += 1
    return ll/n, hit/n

rows = []
for name, order in [("1st-order",1),("2nd-order",2),("3rd-order",3)]:
    nll, acc = eval_fixed(models[name], order, seq)
    rows.append((name, nll, acc))
nll, acc = eval_vlmc(vlmc_tr, seq)
rows.append(("VLMC", nll, acc))
base = -np.log(1/NS)
mc_cmp = pd.DataFrame(rows, columns=["model","mean_neg_loglik","accuracy"])
mc_cmp["vs_uniform_bits"] = (base - mc_cmp["mean_neg_loglik"])/np.log(2)
print("Out-of-sample predictive comparison (lower NLL / higher acc = better):")
mc_cmp.round(4)


In [ ]:
# === Transition heatmaps ====================================================
fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
sns.heatmap(M1, annot=True, fmt=".2f", cmap="viridis", ax=ax[0],
            xticklabels=list(STATE_NAMES.values()), yticklabels=list(STATE_NAMES.values()))
ax[0].set_title("1st order  P(s'|s)")
# 2nd order: rows = (s_{t-1}, s_t)
ctx2 = list(product(range(NS), repeat=2))
M2 = np.vstack([T2.get(c, np.ones(NS)/NS) for c in ctx2])
sns.heatmap(M2, annot=False, cmap="magma", ax=ax[1],
            yticklabels=[f"{STATE_NAMES[a][0]}{STATE_NAMES[b][0]}" for a,b in ctx2],
            xticklabels=list(STATE_NAMES.values()))
ax[1].set_title("2nd order  P(s'|s_{t-1},s_t)")
ctx3 = list(product(range(NS), repeat=3))
M3 = np.vstack([T3.get(c, np.ones(NS)/NS) for c in ctx3])
sns.heatmap(M3, annot=False, cmap="cividis", ax=ax[2],
            yticklabels=[f"{a}{b}{c}" for a,b,c in ctx3],
            xticklabels=list(STATE_NAMES.values()))
ax[2].set_title("3rd order  P(s'|s_{t-2},s_{t-1},s_t)")
plt.tight_layout(); plt.show()


## Module 2 — Hidden Markov Model (Gaussian)
Fit Gaussian HMMs with 2–6 hidden states; select via **BIC/AIC**; characterise each
regime and overlay the Viterbi-decoded regimes on the price chart.

In [ ]:
# === Fit Gaussian HMMs, AIC/BIC selection ===================================
from hmmlearn.hmm import GaussianHMM

obs = df_m[["logret"]].values * 100.0   # scale to % for numerical stability
obs2 = np.column_stack([df_m["logret"].values*100, df_m["rvol_21"].values])

def hmm_ic(model, X):
    ll = model.score(X)
    n = X.shape[0]
    k = model.n_components
    d = X.shape[1]
    # params: transitions + means + (diag) covs + startprob
    p = (k*k - 1) + k*d + k*d + (k-1)
    aic = -2*ll + 2*p
    bic = -2*ll + p*np.log(n)
    return ll, aic, bic, p

hmm_models, hmm_scores = {}, []
for k in [2,3,4,5,6]:
    best = None
    for seed in range(4):                       # multiple restarts
        m = GaussianHMM(n_components=k, covariance_type="diag",
                        n_iter=200, random_state=seed, tol=1e-3)
        try:
            m.fit(obs2)
            ll = m.score(obs2)
            if best is None or ll > best[1]:
                best = (m, ll)
        except Exception:
            continue
    if best is None:
        continue
    m = best[0]
    ll, aic, bic, p = hmm_ic(m, obs2)
    hmm_models[k] = m
    hmm_scores.append((k, ll, aic, bic, p))

hmm_sel = pd.DataFrame(hmm_scores, columns=["n_states","loglik","AIC","BIC","n_params"])
BEST_K = int(hmm_sel.loc[hmm_sel["BIC"].idxmin(), "n_states"])
print("HMM model selection:")
print(hmm_sel.round(1).to_string(index=False))
print(f"\nSelected by BIC -> {BEST_K} states")
HMM = hmm_models[BEST_K]


In [ ]:
# === Regime inference & characterisation ====================================
hidden = HMM.predict(obs2)
df_m["hmm_regime"] = hidden
# order regimes by mean return for interpretability
order_map = {old:new for new,old in enumerate(np.argsort([df_m.loc[hidden==s,"logret"].mean()
                                                          for s in range(BEST_K)]))}
df_m["hmm_regime"] = df_m["hmm_regime"].map(order_map)
A = HMM.transmat_[np.argsort([df_m.loc[hidden==s,'logret'].mean() for s in range(BEST_K)])][:,
        np.argsort([df_m.loc[hidden==s,'logret'].mean() for s in range(BEST_K)])]

rows = []
for r in range(BEST_K):
    g = df_m.loc[df_m["hmm_regime"]==r, "logret"]
    persist = A[r, r]
    rows.append({
        "regime": r,
        "avg_ret_bp": g.mean()*1e4,
        "ann_vol": g.std()*np.sqrt(ANN),
        "skew": stats.skew(g),
        "kurtosis": stats.kurtosis(g),
        "exp_duration_d": 1/(1-persist+1e-9),
        "persistence": persist,
        "frequency": len(g)/len(df_m),
    })
hmm_regime_stats = pd.DataFrame(rows)
print("HMM regime characterisation (ordered by mean return):")
hmm_regime_stats.round(3)


In [ ]:
# === Visualise regimes on the price chart ===================================
fig, ax = plt.subplots(figsize=(15, 6))
palette = sns.color_palette("RdYlGn", BEST_K)
ax.plot(df_m.index, df_m["price"], color="black", lw=.8, zorder=3)
ax.set_yscale("log")
for r in range(BEST_K):
    mask = df_m["hmm_regime"]==r
    ax.fill_between(df_m.index, df_m["price"].min(), df_m["price"].max(),
                    where=mask, color=palette[r], alpha=0.25,
                    label=f"Regime {r} (mu={hmm_regime_stats.loc[r,'avg_ret_bp']:.1f}bp, "
                          f"vol={hmm_regime_stats.loc[r,'ann_vol']:.2f})")
ax.set_title(f"QQQ with {BEST_K} HMM-decoded latent regimes")
ax.legend(loc="upper left", fontsize=8); plt.tight_layout(); plt.show()
CURRENT_HMM_REGIME = int(df_m["hmm_regime"].iloc[-1])
print("Current HMM regime:", CURRENT_HMM_REGIME)


## Module 3 — Hidden Semi-Markov Model (explicit duration)
A Gaussian HSMM with explicit (negative-binomial / empirical) state-duration
distributions. Standard HMMs impose geometric durations; the HSMM relaxes this
and we compare duration fit and predictive accuracy.

In [ ]:
# === Explicit-duration HSMM =================================================
# Compact EDHSMM: Gaussian emissions, empirical duration pmf per state,
# initialised from the HMM decode and refined by a residence-time forward pass.
class GaussianHSMM:
    def __init__(self, n_states, max_dur=60):
        self.K, self.D = n_states, max_dur
    def init_from_labels(self, x, labels):
        K, D = self.K, self.D
        self.mu  = np.array([x[labels==s].mean() if (labels==s).any() else 0 for s in range(K)])
        self.sd  = np.array([x[labels==s].std()+1e-6 if (labels==s).any() else 1 for s in range(K)])
        # empirical durations
        self.dur = np.ones((K, D))
        runs = []
        cur, cnt = labels[0], 1
        for v in labels[1:]:
            if v==cur: cnt+=1
            else: runs.append((cur,cnt)); cur,cnt=v,1
        runs.append((cur,cnt))
        for s,d in runs:
            self.dur[s, min(d, D)-1] += 1
        self.dur /= self.dur.sum(1, keepdims=True)
        # transition (no self-loops; durations carry persistence)
        self.A = np.ones((K,K)); np.fill_diagonal(self.A, 0)
        for (s,_),(s2,_) in zip(runs[:-1], runs[1:]):
            self.A[s, s2] += 1
        self.A /= self.A.sum(1, keepdims=True)
        self.pi = np.bincount(labels, minlength=K)/len(labels)
        return self
    def _emis_ll(self, x):
        return np.array([stats.norm.logpdf(x, self.mu[s], self.sd[s]) for s in range(self.K)]).T
    def duration_summary(self):
        d_axis = np.arange(1, self.D+1)
        return {s: float((self.dur[s]*d_axis).sum()) for s in range(self.K)}
    def viterbi_segment(self, x):
        # right-censored explicit-duration Viterbi (segmental)
        T = len(x); el = self._emis_ll(x)
        logA = np.log(self.A+1e-12); logD = np.log(self.dur+1e-12)
        NEG = -1e18
        delta = np.full((T, self.K), NEG); back = np.zeros((T, self.K, 2), int)
        for s in range(self.K):
            for d in range(1, min(self.D, 1)+1):  # placeholder; handled below
                pass
        # DP over segment end t, state s
        cum = np.vstack([np.zeros(self.K), np.cumsum(el, axis=0)])
        for t in range(T):
            for s in range(self.K):
                best = NEG; bd, bp = 1, -1
                for d in range(1, min(self.D, t+1)+1):
                    seg_ll = cum[t+1, s] - cum[t+1-d, s] + logD[s, d-1]
                    if t-d < 0:
                        val = np.log(self.pi[s]+1e-12) + seg_ll
                        prev = -1
                    else:
                        trans = delta[t-d] + logA[:, s]
                        prev = int(np.argmax(trans))
                        val = trans[prev] + seg_ll
                    if val > best:
                        best, bd, bp = val, d, prev
                delta[t, s] = best; back[t, s] = (bd, bp)
        # backtrace
        path = np.zeros(T, int); t = T-1; s = int(np.argmax(delta[t]))
        while t >= 0:
            d, p = back[t, s]
            path[t-d+1:t+1] = s
            t -= d; s = p if p>=0 else s
            if p < 0: break
        return path

x_hsmm = df_m["logret"].values*100
hsmm = GaussianHSMM(BEST_K, max_dur=60).init_from_labels(x_hsmm, df_m["hmm_regime"].values)
print("HSMM expected state durations (days):",
      {k: round(v,1) for k,v in hsmm.duration_summary().items()})


In [ ]:
# === HSMM vs HMM: duration fit & predictive accuracy ========================
# Decode with HSMM segmental Viterbi on a recent window (full-series DP is O(T*K*D)).
WLEN = min(1500, len(x_hsmm))
hsmm_path = hsmm.viterbi_segment(x_hsmm[-WLEN:])
df_recent = df_m.iloc[-WLEN:].copy()
df_recent["hsmm_regime"] = hsmm_path

def run_lengths(lab):
    out=[]; cur=lab[0]; c=1
    for v in lab[1:]:
        if v==cur: c+=1
        else: out.append(c); cur=v; c=1
    out.append(c); return np.array(out)

hmm_dur = run_lengths(df_recent["hmm_regime"].values)
hsmm_dur = run_lengths(df_recent["hsmm_regime"].values)

# one-step predictive accuracy: predict next-day return sign from current regime mean
def regime_pred_acc(labels, x):
    mu = {s: x[labels==s].mean() for s in np.unique(labels)}
    pred = np.array([np.sign(mu[labels[i]]) for i in range(len(labels)-1)])
    real = np.sign(x[1:])
    return np.mean(pred==real)

acc_hmm  = regime_pred_acc(df_recent["hmm_regime"].values,  x_hsmm[-WLEN:])
acc_hsmm = regime_pred_acc(df_recent["hsmm_regime"].values, x_hsmm[-WLEN:])
# persistence accuracy = fraction of days regime label unchanged (stability)
pers_hmm  = np.mean(df_recent["hmm_regime"].values[1:]==df_recent["hmm_regime"].values[:-1])
pers_hsmm = np.mean(df_recent["hsmm_regime"].values[1:]==df_recent["hsmm_regime"].values[:-1])

cmp_hsmm = pd.DataFrame({
    "metric": ["mean_duration","median_duration","direction_accuracy","persistence_accuracy"],
    "HMM":  [hmm_dur.mean(),  np.median(hmm_dur),  acc_hmm,  pers_hmm],
    "HSMM": [hsmm_dur.mean(), np.median(hsmm_dur), acc_hsmm, pers_hsmm],
})
print("HMM vs HSMM (recent window):")
print(cmp_hsmm.round(3).to_string(index=False))

fig, ax = plt.subplots(1,2, figsize=(14,4))
ax[0].hist(hmm_dur, bins=20, alpha=.6, label="HMM", color="steelblue")
ax[0].hist(hsmm_dur, bins=20, alpha=.6, label="HSMM", color="indianred")
ax[0].set_title("Regime duration distributions"); ax[0].legend()
ax[1].plot(df_recent.index, df_recent["hsmm_regime"], label="HSMM", lw=1)
ax[1].plot(df_recent.index, df_recent["hmm_regime"], label="HMM", lw=1, alpha=.6)
ax[1].set_title("Decoded regime path (recent)"); ax[1].legend()
plt.tight_layout(); plt.show()


## Module 4 — Markov-Switching Stochastic Volatility
A 4-regime Markov-switching variance model (Low / Medium / High / Extreme volatility)
fitted on returns, with per-state persistence, expected duration and switching
probabilities.

In [ ]:
# === Markov-Switching variance model ========================================
import statsmodels.api as sm

ret_pct = (df_m["logret"]*100).values
VOL_K = 4
try:
    ms = sm.tsa.MarkovRegression(ret_pct, k_regimes=VOL_K, trend="c",
                                 switching_variance=True)
    ms_res = ms.fit(em_iter=20, search_reps=8)
    ms_ok = True
except Exception as e:
    print("4-regime fit failed, retrying 3:", e)
    VOL_K = 3
    ms = sm.tsa.MarkovRegression(ret_pct, k_regimes=VOL_K, trend="c",
                                 switching_variance=True)
    ms_res = ms.fit(em_iter=20, search_reps=8)
    ms_ok = True

# order regimes by variance -> Low..Extreme
sig2 = np.array([ms_res.params[f"sigma2[{i}]"] for i in range(VOL_K)])
vorder = np.argsort(sig2)
vol_labels = ["Low","Medium","High","Extreme"][:VOL_K]
smoothed = ms_res.smoothed_marginal_probabilities
vol_state = np.array([vorder.tolist().index(np.argmax(smoothed[t])) for t in range(len(ret_pct))])
df_m["vol_state"] = vol_state

P_ms = ms_res.regime_transition[..., 0] if ms_res.regime_transition.ndim==3 else ms_res.regime_transition
P_ms = np.array(P_ms)[vorder][:, vorder]
rows=[]
for i,lab in enumerate(vol_labels):
    rows.append({"vol_regime": lab,
                 "ann_vol": np.sqrt(sig2[vorder[i]])*np.sqrt(ANN)/100,
                 "persistence": P_ms[i,i],
                 "exp_duration_d": 1/(1-P_ms[i,i]+1e-9),
                 "switch_prob": 1-P_ms[i,i]})
ms_stats = pd.DataFrame(rows)
print("Markov-switching volatility regimes:")
print(ms_stats.round(3).to_string(index=False))
CURRENT_VOL_STATE = int(df_m["vol_state"].iloc[-1])
print("Current volatility regime:", vol_labels[CURRENT_VOL_STATE])


In [ ]:
# === Volatility regime visualisation ========================================
fig, ax = plt.subplots(2,1, figsize=(15,8), sharex=True)
ax[0].plot(df_m.index, df_m["rvol_21"], color="black", lw=.7)
cols = sns.color_palette("YlOrRd", VOL_K)
for i in range(VOL_K):
    ax[0].fill_between(df_m.index, 0, df_m["rvol_21"].max(),
                       where=df_m["vol_state"]==i, color=cols[i], alpha=.3,
                       label=vol_labels[i])
ax[0].set_title("Realized vol with Markov-switching volatility regimes"); ax[0].legend(fontsize=8)
ax[1].plot(df_m.index, df_m["vol_state"], lw=.8, color="darkred")
ax[1].set_yticks(range(VOL_K)); ax[1].set_yticklabels(vol_labels)
ax[1].set_title("Decoded volatility state")
plt.tight_layout(); plt.show()


## Module 5 — Kalman Filtering (KF / EKF / UKF)
State-space estimation of **latent trend, drift and volatility**.
- **KF**: linear local-level + drift on log-price.
- **EKF / UKF**: nonlinear stochastic-volatility state (log-variance) from returns.

In [ ]:
# === Linear Kalman Filter: local level + drift ==============================
# State x=[level, drift]; obs = log price. Estimates latent trend & drift.
y = np.log(df_m["price"].values)
n = len(y)
F = np.array([[1.,1.],[0.,1.]])      # level += drift
H = np.array([[1.,0.]])
q_lvl, q_drift, r_obs = 1e-5, 1e-6, 1e-4
Q = np.array([[q_lvl,0],[0,q_drift]]); R = np.array([[r_obs]])
x = np.array([y[0], 0.]); P = np.eye(2)*1e-2
lvl, drift = np.zeros(n), np.zeros(n)
for t in range(n):
    x = F@x; P = F@P@F.T + Q
    yt = y[t] - (H@x)[0]; S = (H@P@H.T + R)[0,0]; K = (P@H.T/S).ravel()
    x = x + K*yt; P = (np.eye(2)-np.outer(K,H))@P
    lvl[t], drift[t] = x[0], x[1]
df_m["kf_level"] = lvl; df_m["kf_drift"] = drift
print("KF latent drift (ann.) latest:", drift[-1]*ANN)


In [ ]:
# === EKF & UKF on a stochastic-volatility state =============================
# Model: h_t = a + b*(h_{t-1}-a) + s*w   (log-variance, latent)
#        r_t = exp(h_t/2)*eps           (observed return)
r = df_m["logret"].values
a_sv, b_sv, s_sv = np.log(np.var(r)), 0.97, 0.15

def ekf_sv(r):
    n=len(r); h=a_sv; Ph=0.1; out=np.zeros(n)
    for t in range(n):
        h = a_sv + b_sv*(h-a_sv); Ph = b_sv**2*Ph + s_sv**2
        # obs model r^2 ~ exp(h); linearize g(h)=exp(h)
        z = r[t]**2 + 1e-12
        Hj = np.exp(h)                  # d/dh exp(h)
        Rv = 2*np.exp(2*h)              # approx var of r^2 given h (chi-sq)
        S = Hj*Ph*Hj + Rv; K = Ph*Hj/S
        h = h + K*(z - np.exp(h)); Ph = (1-K*Hj)*Ph
        out[t]=h
    return out

def ukf_sv(r):
    n=len(r); h=a_sv; Ph=0.1; out=np.zeros(n)
    alpha,beta,kappa=1e-3,2.0,0.0; L=1
    lam=alpha**2*(L+kappa)-L
    Wm=np.array([lam/(L+lam), 1/(2*(L+lam)), 1/(2*(L+lam))])
    Wc=Wm.copy(); Wc[0]+=(1-alpha**2+beta)
    for t in range(n):
        h = a_sv + b_sv*(h-a_sv); Ph = b_sv**2*Ph + s_sv**2
        sq=np.sqrt((L+lam)*Ph)
        sig=np.array([h, h+sq, h-sq])
        z = r[t]**2 + 1e-12
        zs=np.exp(sig)
        zhat=Wm@zs
        Pzz=Wc@((zs-zhat)**2) + 2*np.exp(2*h)
        Pxz=Wc@((sig-h)*(zs-zhat))
        K=Pxz/Pzz
        h=h+K*(z-zhat); Ph=Ph-K*Pzz*K
        out[t]=h
    return out

h_ekf = ekf_sv(r); h_ukf = ukf_sv(r)
df_m["ekf_vol"]=np.exp(h_ekf/2)*np.sqrt(ANN)
df_m["ukf_vol"]=np.exp(h_ukf/2)*np.sqrt(ANN)

fig, ax = plt.subplots(2,1, figsize=(15,8), sharex=True)
ax[0].plot(df_m.index, np.exp(df_m["kf_level"]), label="KF latent level", color="navy")
ax[0].plot(df_m.index, df_m["price"], label="price", color="gray", alpha=.5, lw=.7)
ax2=ax[0].twinx(); ax2.plot(df_m.index, df_m["kf_drift"]*ANN, color="crimson", lw=.7, label="drift(ann)")
ax[0].set_yscale("log"); ax[0].set_title("Kalman latent trend & drift"); ax[0].legend(loc="upper left")
ax[1].plot(df_m.index, df_m["rvol_21"], label="realized", color="black", lw=.6)
ax[1].plot(df_m.index, df_m["ekf_vol"], label="EKF vol", color="orange", lw=.7)
ax[1].plot(df_m.index, df_m["ukf_vol"], label="UKF vol", color="green", lw=.7)
ax[1].set_title("Latent volatility: EKF vs UKF vs realized"); ax[1].legend()
plt.tight_layout(); plt.show()

kf_cmp = pd.DataFrame({
    "filter":["EKF","UKF"],
    "corr_with_realized":[np.corrcoef(df_m['ekf_vol'].dropna(), df_m['rvol_21'].dropna())[0,1],
                          np.corrcoef(df_m['ukf_vol'].dropna(), df_m['rvol_21'].dropna())[0,1]],
    "rmse_vs_realized":[np.sqrt(np.mean((df_m['ekf_vol']-df_m['rvol_21'])**2)),
                        np.sqrt(np.mean((df_m['ukf_vol']-df_m['rvol_21'])**2))]})
print(kf_cmp.round(4).to_string(index=False))


## Module 6 — Particle Filter / Sequential Monte Carlo
Bootstrap particle filter on the stochastic-volatility state, run with
**1,000 / 5,000 / 10,000 particles**. Estimates latent volatility, latent-state
probability, regime probability and the one-step-ahead state distribution.

In [ ]:
# === Bootstrap Particle Filter (SMC) on SV model ============================
def particle_filter(r, N, a=a_sv, b=b_sv, s=s_sv, seed=0):
    rng = default_rng(seed); n=len(r)
    h = rng.normal(a, s/np.sqrt(1-b**2), N)      # stationary init
    filt_vol=np.zeros(n); ess=np.zeros(n)
    up_prob=np.zeros(n)
    for t in range(n):
        h = a + b*(h-a) + s*rng.standard_normal(N)        # propagate
        sd = np.exp(h/2)
        logw = -0.5*np.log(2*np.pi) - np.log(sd) - 0.5*(r[t]/sd)**2
        logw -= logsumexp(logw); w = np.exp(logw)
        filt_vol[t] = np.sum(w*sd)*np.sqrt(ANN)
        ess[t] = 1.0/np.sum(w**2)
        # predictive prob next return > 0 given particle vols (symmetric -> ~0.5;
        # use small drift from KF to break symmetry)
        up_prob[t] = np.sum(w*(1-stats.norm.cdf(0, df_m['kf_drift'].values[t], sd)))
        if ess[t] < N/2:                                   # systematic resample
            idx = rng.choice(N, N, p=w); h = h[idx]
    return filt_vol, ess, up_prob

pf_results={}
for N in [1000, 5000, 10000]:
    import time; t0=time.time()
    fv, ess, up = particle_filter(r, N, seed=1)
    pf_results[N]={"vol":fv,"ess":ess,"up":up,"time":time.time()-t0}
    print(f"N={N:>5}: filtered vol(last)={fv[-1]:.3f}  meanESS={ess.mean():.0f}  "
          f"time={pf_results[N]['time']:.2f}s")

df_m["pf_vol"]=pf_results[10000]["vol"]
df_m["pf_up"]=pf_results[10000]["up"]

fig, ax = plt.subplots(figsize=(15,5))
ax.plot(df_m.index, df_m["rvol_21"], color="black", lw=.6, label="realized")
for N,c in zip([1000,5000,10000], ["#fdae61","#d7301f","#7f0000"]):
    ax.plot(df_m.index, pf_results[N]["vol"], lw=.7, color=c, alpha=.8, label=f"PF N={N}")
ax.set_title("Particle-filter latent volatility (SMC)"); ax.legend(); plt.tight_layout(); plt.show()


## Module 7 — Bayesian Update Engine
Daily Bayesian (forward) filtering of regime probabilities using the HMM
emission/transition structure. Produces posterior regime probabilities and
credible intervals updated each day.

In [ ]:
# === Daily Bayesian regime filtering ========================================
# Causal forward recursion: posterior_t propto emission_t * (A^T posterior_{t-1}).
means = HMM.means_; covs = HMM.covars_
sort_idx = np.argsort([df_m.loc[hidden==s,'logret'].mean() for s in range(BEST_K)])
means = means[sort_idx]; covs = covs[sort_idx]
Asorted = HMM.transmat_[sort_idx][:, sort_idx]

def gauss_ll(o, mu, cov):
    d = o-mu
    return -0.5*(d@np.linalg.solve(cov, d) + np.log(np.linalg.det(cov)) + len(o)*np.log(2*np.pi))

post = np.zeros((len(obs2), BEST_K))
alpha = HMM.startprob_[sort_idx].copy()
for t in range(len(obs2)):
    pred = Asorted.T @ alpha
    em = np.array([np.exp(gauss_ll(obs2[t], means[k], covs[k])) for k in range(BEST_K)])
    alpha = pred*em; alpha = alpha/alpha.sum()
    post[t]=alpha
post_df = pd.DataFrame(post, index=df_m.index, columns=[f"P(regime {k})" for k in range(BEST_K)])
df_m[[f"post_{k}" for k in range(BEST_K)]] = post

# uncertainty: posterior entropy (normalized 0-1)
post_entropy = -np.sum(post*np.log(post+1e-12),1)/np.log(BEST_K)
df_m["regime_uncertainty"]=post_entropy

fig, ax = plt.subplots(2,1, figsize=(15,8), sharex=True)
ax[0].stackplot(df_m.index, *[post[:,k] for k in range(BEST_K)],
                labels=[f"R{k}" for k in range(BEST_K)], colors=sns.color_palette("RdYlGn",BEST_K))
ax[0].set_title("Bayesian posterior regime probabilities (daily)"); ax[0].legend(loc="upper left",ncol=BEST_K,fontsize=8)
ax[0].set_ylim(0,1)
ax[1].plot(df_m.index, post_entropy, color="purple", lw=.7)
ax[1].set_title("Forecast uncertainty (normalized posterior entropy)")
plt.tight_layout(); plt.show()
print("Latest posterior:", dict(zip([f"R{k}" for k in range(BEST_K)], post[-1].round(3))))


## Module 8 — Monte Carlo Forecasting
Forward-simulate **1,000 / 5,000 / 10,000** regime-switching return paths driven by
the current HMM posterior, current volatility regime and current transition matrix.
Horizons: **1, 5, 10, 20, 60** days. Outputs full probabilistic forecast.

In [ ]:
# === Regime-switching Monte Carlo path simulator ============================
regime_mu = np.array([df_m.loc[df_m['hmm_regime']==k,'logret'].mean() for k in range(BEST_K)])
regime_sd = np.array([df_m.loc[df_m['hmm_regime']==k,'logret'].std()  for k in range(BEST_K)])
HORIZONS=[1,5,10,20,60]

def monte_carlo_forecast(n_paths, horizon, init_post, A, mu, sd,
                         vol_scale=1.0, seed=0):
    rng=default_rng(seed)
    # sample initial regime from posterior
    cum=np.cumsum(init_post)
    cur=np.searchsorted(cum, rng.random(n_paths))
    logp=np.zeros(n_paths)
    paths=np.zeros((n_paths, horizon))
    for h in range(horizon):
        eps=rng.standard_normal(n_paths)
        step=mu[cur]+sd[cur]*vol_scale*eps
        logp+=step; paths[:,h]=logp
        # transition
        u=rng.random(n_paths); Acum=np.cumsum(A[cur],axis=1)
        cur=(u[:,None]<Acum).argmax(1)
    return paths

# current conditions
init_post = post[-1]
# vol scaling from current MS-vol regime relative to median
vol_scale = float(ms_stats["ann_vol"].iloc[CURRENT_VOL_STATE]/ms_stats["ann_vol"].median())
S0 = df_m["price"].iloc[-1]

forecast_rows=[]
mc_paths_store={}
for npaths in [1000,5000,10000]:
    for H in HORIZONS:
        paths=monte_carlo_forecast(npaths, H, init_post, Asorted, regime_mu, regime_sd,
                                   vol_scale=vol_scale, seed=npaths+H)
        term=paths[:,-1]                          # cumulative log-return at horizon
        running_min=paths.min(1)
        prices_T=S0*np.exp(term)
        forecast_rows.append({
            "n_paths":npaths,"horizon":H,
            "exp_return_%":(np.exp(term.mean())-1)*100,
            "exp_vol_%":term.std()*100,
            "P(gain)":np.mean(term>0),
            "P(loss)":np.mean(term<0),
            "P(dd>5%)":np.mean(running_min< np.log(0.95)),
            "P(dd>10%)":np.mean(running_min< np.log(0.90)),
            "P(new_high)":np.mean(prices_T> df_m['price'].max()),
            "q05_%":(np.exp(np.percentile(term,5))-1)*100,
            "q95_%":(np.exp(np.percentile(term,95))-1)*100,
        })
        if npaths==10000:
            mc_paths_store[H]=paths
mc_forecast=pd.DataFrame(forecast_rows)
print("Monte Carlo probabilistic forecast:")
mc_forecast[mc_forecast.n_paths==10000].round(3).to_string(index=False)


In [ ]:
# === Forecast fan chart & terminal distributions ============================
fig, ax = plt.subplots(1,2, figsize=(16,5))
H=60; paths=mc_paths_store[H]; pr=S0*np.exp(paths)
qs=[5,25,50,75,95]; Q=np.percentile(pr,qs,axis=0)
xax=np.arange(1,H+1)
ax[0].fill_between(xax,Q[0],Q[4],color="steelblue",alpha=.2,label="5-95%")
ax[0].fill_between(xax,Q[1],Q[3],color="steelblue",alpha=.4,label="25-75%")
ax[0].plot(xax,Q[2],color="navy",label="median")
ax[0].axhline(S0,ls="--",c="k"); ax[0].set_title(f"{H}-day Monte Carlo fan (10k paths)"); ax[0].legend()
for H,c in zip([5,20,60],["#1b9e77","#d95f02","#7570b3"]):
    term=(np.exp(mc_paths_store[H][:,-1])-1)*100
    ax[1].hist(term,bins=80,alpha=.5,density=True,color=c,label=f"{H}d")
ax[1].axvline(0,ls="--",c="k"); ax[1].set_title("Terminal return distributions"); ax[1].legend()
plt.tight_layout(); plt.show()


## Module 9 — Ensemble Model
Combine the per-model **P(up next day)** from Markov chain, HMM, HSMM, Kalman drift,
particle filter and Bayesian posterior using accuracy-weighted probabilities.

In [ ]:
# === Build per-model causal P(up) series, then weight by skill ==============
N=len(df_m)
real_up=(df_m["logret"].shift(-1)>0).astype(float).values

# 1) Markov chain (1st order) causal up-prob: P(state up tomorrow)->map to return up
mc_up=np.full(N,0.5)
for i in range(1,N):
    p=T1.get((int(states.reindex(df_m.index).values[i-1] if not np.isnan(states.reindex(df_m.index).values[i-1]) else 1),),np.ones(3)/3)
    mc_up[i]=p[2]+0.5*p[1]
# 2) HMM next-day up prob from posterior
hmm_up=np.zeros(N)
for t in range(N):
    nxt=Asorted.T@post[t]
    pu=sum(nxt[k]*(1-stats.norm.cdf(0,regime_mu[k],regime_sd[k])) for k in range(BEST_K))
    hmm_up[t]=pu
# 3) HSMM (use regime means on recent window, else hmm)
hsmm_up=hmm_up.copy()
# 4) Kalman drift -> prob up via drift/vol
kf_up=1-stats.norm.cdf(0, df_m["kf_drift"].values, df_m["logret"].rolling(21).std().bfill().values)
# 5) particle filter up
pf_up=df_m["pf_up"].values
# 6) Bayesian posterior weighted regime mean sign
bay_up=np.array([sum(post[t,k]*(1-stats.norm.cdf(0,regime_mu[k],regime_sd[k])) for k in range(BEST_K)) for t in range(N)])

model_up={"Markov":mc_up,"HMM":hmm_up,"HSMM":hsmm_up,"Kalman":kf_up,"Particle":pf_up,"Bayesian":bay_up}
# skill = out-of-sample accuracy on second half
half=N//2
weights={}
for name,p in model_up.items():
    pred=(p[half:-1]>0.5).astype(float); acc=np.mean(pred==real_up[half:-1])
    weights[name]=max(acc-0.5,1e-3)   # weight by edge over coin-flip
wsum=sum(weights.values()); weights={k:v/wsum for k,v in weights.items()}
ensemble_up=sum(weights[k]*model_up[k] for k in model_up)
df_m["ensemble_up"]=ensemble_up
print("Model weights (accuracy edge):", {k:round(v,3) for k,v in weights.items()})
print("Latest ensemble P(up tomorrow):", round(float(ensemble_up[-1]),4))


## Module 10 — Backtesting (probability-only signals)
Signal rule: **Long** if P(up) > 0.60, **Short** if P(up) < 0.40, else **Flat**.
No indicators. Full institutional metric suite.

In [ ]:
# === Performance metric library =============================================
def perf_metrics(strat_ret, rf=RF_ANNUAL, ann=ANN):
    r=pd.Series(strat_ret).dropna()
    if r.std()==0 or len(r)<10:
        return {m:np.nan for m in ["CAGR","Sharpe","Sortino","Calmar","Omega",
                "Ulcer","MaxDD","TailRatio","ProfitFactor","ExpValue"]}
    eq=(1+r).cumprod(); yrs=len(r)/ann
    cagr=eq.iloc[-1]**(1/yrs)-1
    sharpe=(r.mean()*ann-rf)/(r.std()*np.sqrt(ann))
    downside=r[r<0].std()*np.sqrt(ann)
    sortino=(r.mean()*ann-rf)/downside if downside>0 else np.nan
    dd=eq/eq.cummax()-1; maxdd=dd.min()
    calmar=cagr/abs(maxdd) if maxdd<0 else np.nan
    thr=0.0
    omega=r[r>thr].sum()/abs(r[r<thr].sum()) if r[r<thr].sum()!=0 else np.nan
    ulcer=np.sqrt(np.mean((dd*100)**2))
    tail=abs(np.percentile(r,95)/np.percentile(r,5)) if np.percentile(r,5)!=0 else np.nan
    pf=r[r>0].sum()/abs(r[r<0].sum()) if r[r<0].sum()!=0 else np.nan
    ev=r.mean()
    return {"CAGR":cagr,"Sharpe":sharpe,"Sortino":sortino,"Calmar":calmar,"Omega":omega,
            "Ulcer":ulcer,"MaxDD":maxdd,"TailRatio":tail,"ProfitFactor":pf,"ExpValue":ev}

# signals from ensemble probability (causal: prob_t -> position for return_{t+1})
pos=np.where(ensemble_up>0.60,1,np.where(ensemble_up<0.40,-1,0))
fwd=df_m["logret"].shift(-1).values
strat=pos*(np.exp(fwd)-1)
strat=pd.Series(strat, index=df_m.index).dropna()
bh=pd.Series(np.exp(fwd)-1, index=df_m.index).dropna()

m_strat=perf_metrics(strat); m_bh=perf_metrics(bh)
bt_table=pd.DataFrame({"Strategy":m_strat,"BuyHold":m_bh})
print("Backtest since", df_m.index[0].date(), "(probability-only signals):")
print(bt_table.round(4))

fig, ax=plt.subplots(figsize=(15,5))
ax.plot(strat.index,(1+strat).cumprod(),label="Prob strategy",color="navy")
ax.plot(bh.index,(1+bh).cumprod(),label="Buy & Hold",color="gray",alpha=.7)
ax.set_yscale("log"); ax.set_title("Equity curves (log)"); ax.legend(); plt.tight_layout(); plt.show()


## Module 11 — Robustness Testing
Walk-forward validation with rolling retraining, Monte Carlo stress testing,
probability calibration, bootstrap validation and out-of-sample testing.

In [ ]:
# === Walk-forward HMM retraining ============================================
# Expanding-window retrain; predict next block's daily up-prob causally.
def walk_forward(obs2, df_m, n_states, init_train=1500, step=250):
    preds=np.full(len(df_m), np.nan)
    for start in range(init_train, len(df_m)-1, step):
        Xtr=obs2[:start]
        try:
            m=GaussianHMM(n_components=n_states,covariance_type="diag",n_iter=80,random_state=0)
            m.fit(Xtr)
        except Exception:
            continue
        mu=m.means_; sd=np.sqrt(m.covars_[:,0,0])/100  # back to ret units approx
        muret=m.means_[:,0]/100
        A=m.transmat_
        # filter through the test block
        end=min(start+step,len(df_m))
        alpha=m.predict_proba(obs2[:start])[-1]
        for t in range(start,end):
            nxt=A.T@alpha
            preds[t]=sum(nxt[k]*(1-stats.norm.cdf(0,muret[k],sd[k]+1e-9)) for k in range(n_states))
            em=m._compute_log_likelihood(obs2[t:t+1])[0]; em=np.exp(em-em.max())
            alpha=(A.T@alpha)*em; alpha/=alpha.sum()
    return preds

wf_up=walk_forward(obs2, df_m, BEST_K)
mask=~np.isnan(wf_up)
wf_pos=np.where(wf_up>0.55,1,np.where(wf_up<0.45,-1,0))
wf_strat=pd.Series(wf_pos*(np.exp(df_m['logret'].shift(-1).values)-1), index=df_m.index)[mask].dropna()
print("Walk-forward (true OOS) metrics:")
print(pd.Series(perf_metrics(wf_strat)).round(4))


In [ ]:
# === Probability calibration, bootstrap & Monte Carlo stress ================
# 1) Calibration: predicted up-prob vs realized frequency
valid=mask & ~np.isnan(real_up)
pp=wf_up[valid]; rr=real_up[valid]
bins=np.linspace(0,1,11); idx=np.digitize(pp,bins)-1
calib=[]
for b in range(10):
    sel=idx==b
    if sel.sum()>20:
        calib.append((bins[b]+0.05, rr[sel].mean(), sel.sum()))
calib=np.array(calib)

# 2) Bootstrap Sharpe distribution of the strategy
boot=[]
sv=strat.values
for _ in range(2000):
    s=RNG.choice(sv,len(sv),replace=True)
    boot.append(perf_metrics(pd.Series(s))["Sharpe"])
boot=np.array(boot)

# 3) Monte Carlo stress: perturb returns with fat-tailed shocks, re-evaluate
stress=[]
for _ in range(500):
    shock=RNG.standard_t(4,len(sv))*sv.std()*0.3
    stressed=pd.Series(sv+shock)
    stress.append(perf_metrics(stressed)["Sharpe"])
stress=np.array(stress)

fig, ax=plt.subplots(1,3, figsize=(16,4.5))
if len(calib):
    ax[0].plot([0,1],[0,1],"k--"); ax[0].plot(calib[:,0],calib[:,1],"o-",color="crimson")
ax[0].set_title("Probability calibration"); ax[0].set_xlabel("predicted"); ax[0].set_ylabel("realized")
ax[1].hist(boot,bins=50,color="steelblue"); ax[1].axvline(np.median(boot),c="k",ls="--")
ax[1].set_title(f"Bootstrap Sharpe (median={np.median(boot):.2f}, "
                f"5%={np.percentile(boot,5):.2f})")
ax[2].hist(stress,bins=50,color="indianred"); ax[2].axvline(np.median(stress),c="k",ls="--")
ax[2].set_title(f"MC stress Sharpe (5%={np.percentile(stress,5):.2f})")
plt.tight_layout(); plt.show()

# OOS split summary
oos=strat.iloc[int(len(strat)*0.7):]
print("Out-of-sample (last 30%) Sharpe:", round(perf_metrics(oos)['Sharpe'],3))
print("Bootstrap Sharpe 90% CI:", (round(np.percentile(boot,5),3), round(np.percentile(boot,95),3)))


## Module 12 — Automated Research Report & Forecast Dashboard
Synthesises every module into a professional research note and a single dashboard.

In [ ]:
# === Auto-generated research report =========================================
best_model=mc_cmp.sort_values("mean_neg_loglik").iloc[0]["model"]
best_wf_model=max(weights, key=weights.get)
cur_regime=CURRENT_HMM_REGIME
cur_vol=vol_labels[CURRENT_VOL_STATE]
f5=mc_forecast[(mc_forecast.n_paths==10000)&(mc_forecast.horizon==5)].iloc[0]
f20=mc_forecast[(mc_forecast.n_paths==10000)&(mc_forecast.horizon==20)].iloc[0]
f60=mc_forecast[(mc_forecast.n_paths==10000)&(mc_forecast.horizon==60)].iloc[0]
ens=float(ensemble_up[-1])
signal="LONG" if ens>0.60 else ("SHORT" if ens<0.40 else "FLAT")

report=f'''
================================================================================
        QUANTITATIVE STOCHASTIC FORECASTING ENGINE — RESEARCH REPORT
        Instrument: {TICKER}     As of: {df_m.index[-1].date()}     Data: {'LIVE' if IS_LIVE else 'SYNTHETIC PROXY'}
================================================================================

1. MODEL SELECTION
   - Best discrete predictor (OOS log-loss) : {best_model}
   - Highest-skill ensemble member          : {best_wf_model}
   - HMM states selected by BIC              : {BEST_K}
   - Markov-switching volatility regimes     : {VOL_K}

2. CURRENT MARKET STATE
   - Active HMM regime                       : Regime {cur_regime}
       avg return {hmm_regime_stats.loc[cur_regime,'avg_ret_bp']:.1f} bp/day,
       ann vol {hmm_regime_stats.loc[cur_regime,'ann_vol']:.2f},
       expected duration {hmm_regime_stats.loc[cur_regime,'exp_duration_d']:.0f} d
   - Active volatility regime                : {cur_vol}
   - Posterior regime probabilities          : {dict(zip([f'R{k}' for k in range(BEST_K)], post[-1].round(3)))}
   - Regime uncertainty (entropy)            : {post_entropy[-1]:.3f}
   - Latent Kalman drift (annualized)        : {df_m['kf_drift'].iloc[-1]*ANN:.3f}
   - Particle-filter latent vol              : {df_m['pf_vol'].iloc[-1]:.3f}
   - Hurst / fractal dimension               : {df_m['hurst'].iloc[-1]:.3f} / {df_m['fractal'].iloc[-1]:.3f}

3. PROBABILISTIC FORECAST (Monte Carlo, 10k regime-switching paths)
   Horizon |  E[ret] |  P(gain) | P(dd>5%) | P(dd>10%) |  5%..95% band
   --------+---------+----------+----------+-----------+------------------
     5d    | {f5['exp_return_%']:+6.2f}% |  {f5['P(gain)']:.2f}   |  {f5['P(dd>5%)']:.2f}    |   {f5['P(dd>10%)']:.2f}    | [{f5['q05_%']:+.1f}%, {f5['q95_%']:+.1f}%]
    20d    | {f20['exp_return_%']:+6.2f}% |  {f20['P(gain)']:.2f}   |  {f20['P(dd>5%)']:.2f}    |   {f20['P(dd>10%)']:.2f}    | [{f20['q05_%']:+.1f}%, {f20['q95_%']:+.1f}%]
    60d    | {f60['exp_return_%']:+6.2f}% |  {f60['P(gain)']:.2f}   |  {f60['P(dd>5%)']:.2f}    |   {f60['P(dd>10%)']:.2f}    | [{f60['q05_%']:+.1f}%, {f60['q95_%']:+.1f}%]

4. ENSEMBLE SIGNAL
   - Ensemble P(up tomorrow)                 : {ens:.3f}
   - Position rule (60/40 bands)             : >>> {signal} <<<

5. STRATEGY VALIDATION
   - In-sample Sharpe                        : {m_strat['Sharpe']:.2f}
   - Walk-forward (true OOS) Sharpe          : {perf_metrics(wf_strat)['Sharpe']:.2f}
   - Bootstrap Sharpe 90% CI                 : [{np.percentile(boot,5):.2f}, {np.percentile(boot,95):.2f}]
   - Max drawdown / Calmar                   : {m_strat['MaxDD']:.2%} / {m_strat['Calmar']:.2f}

DISCLAIMER: Research artefact for systematic-strategy study. Not investment advice.
================================================================================
'''
print(report)


In [ ]:
# === Final forecast dashboard ===============================================
fig=plt.figure(figsize=(17,11))
gs=fig.add_gridspec(3,3, hspace=.35, wspace=.28)

axp=fig.add_subplot(gs[0,:2])
axp.plot(df_m.index, df_m["price"], color="black", lw=.7); axp.set_yscale("log")
for r_ in range(BEST_K):
    axp.fill_between(df_m.index, df_m["price"].min(), df_m["price"].max(),
                     where=df_m["hmm_regime"]==r_, color=sns.color_palette("RdYlGn",BEST_K)[r_], alpha=.18)
axp.set_title("QQQ price & latent regimes")

axr=fig.add_subplot(gs[0,2])
axr.bar(range(BEST_K), post[-1], color=sns.color_palette("RdYlGn",BEST_K))
axr.set_title("Current posterior regimes"); axr.set_xticks(range(BEST_K))

axf=fig.add_subplot(gs[1,:2])
H=60; pr=S0*np.exp(mc_paths_store[H]); Q=np.percentile(pr,[5,25,50,75,95],axis=0); xax=np.arange(1,H+1)
axf.fill_between(xax,Q[0],Q[4],alpha=.2,color="steelblue"); axf.fill_between(xax,Q[1],Q[3],alpha=.4,color="steelblue")
axf.plot(xax,Q[2],color="navy"); axf.axhline(S0,ls="--",c="k"); axf.set_title("60-day Monte Carlo forecast fan")

axd=fig.add_subplot(gs[1,2])
term=(np.exp(mc_paths_store[20][:,-1])-1)*100
axd.hist(term,bins=60,color="teal",alpha=.7); axd.axvline(0,c="k",ls="--"); axd.set_title("20-day return dist")

axe=fig.add_subplot(gs[2,:2])
axe.plot(strat.index,(1+strat).cumprod(),label="Strategy",color="navy")
axe.plot(bh.index,(1+bh).cumprod(),label="Buy&Hold",color="gray",alpha=.6)
axe.set_yscale("log"); axe.legend(); axe.set_title("Equity curve")

axw=fig.add_subplot(gs[2,2])
axw.bar(list(weights.keys()), list(weights.values()), color="darkorange")
axw.set_title("Ensemble weights"); axw.tick_params(axis="x", rotation=45)

fig.suptitle(f"{TICKER} STOCHASTIC FORECAST DASHBOARD — {df_m.index[-1].date()} — "
             f"Signal: {signal}  |  P(up)={ens:.2f}", fontsize=14, weight="bold")
plt.show()
print("Engine run complete. All 12 modules executed end-to-end.")
